# Análisis de Redes Científicas con Grafos

**Área:** Procesamiento de Lenguaje Natural (PLN)

**Asignatura:** Estructuras de Datos — Proyecto Unidad 5

**Equipo:** _(integrantes aquí)_

---

Construimos y analizamos una red de colaboración científica en PLN usando datos reales de la API de Semantic Scholar. Cada nodo es un investigador, cada arista es una co-autoría, y el peso indica el número de publicaciones compartidas.

## 0. Importaciones y configuración

Verifica que todas las librerías estén instaladas. Si alguna falla, ciérralo y corre `pip install <paquete>` en la terminal con el venv activado.

In [ ]:
import json
import time
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import requests
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import community as community_louvain

print('NetworkX:', nx.__version__)
print('Pandas:', pd.__version__)
print('Todo listo.')

## 1. Recolección de datos (10 pts)

**Objetivo:** descargar papers reales de PLN desde la API de Semantic Scholar y guardarlos en JSON local.

**Endpoint:** `https://api.semanticscholar.org/graph/v1/paper/search`

**Datos requeridos por la rúbrica:** títulos, autores, año de publicación.

**Estrategia:** múltiples queries de términos PLN para diversificar y alcanzar >100 autores únicos.

In [ ]:
# --- Configuración ---
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
RAW_PAPERS_PATH = DATA_DIR / 'papers_pln_raw.json'

API_BASE_URL = 'https://api.semanticscholar.org/graph/v1/paper/search'
API_FIELDS = 'title,year,authors.authorId,authors.name'

# Identificarnos ante la API ayuda a evitar throttling agresivo
REQUEST_HEADERS = {
    'User-Agent': 'UADY-EstructurasDatos-Proyecto/1.0 (academic; ri3s@protonmail.com)'
}

PAPERS_PER_QUERY = 100
REQUEST_TIMEOUT_SECONDS = 30
SLEEP_BETWEEN_QUERIES_SECONDS = 8        # conservador para IP compartida
RATE_LIMIT_BACKOFF_BASE_SECONDS = 30     # primer reintento espera 30s, luego 60, 90...
MAX_RETRIES = 5

PLN_QUERIES = [
    'natural language processing',
    'computational linguistics',
    'language model',
    'text classification',
    'named entity recognition',
    'machine translation',
    'sentiment analysis',
    'question answering',
]


def fetch_papers_for_query(query: str, limit: int = PAPERS_PER_QUERY) -> list | None:
    """Descarga papers de Semantic Scholar para una query.

    Retorna la lista de papers, o None si la API falló después de los reintentos.
    """
    params = {'query': query, 'limit': limit, 'fields': API_FIELDS}

    for retry_index in range(MAX_RETRIES):
        response = requests.get(
            API_BASE_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        if response.status_code == 200:
            return response.json().get('data', [])

        if response.status_code == 429:
            wait_seconds = RATE_LIMIT_BACKOFF_BASE_SECONDS * (retry_index + 1)
            print(f'  Rate limit alcanzado, esperando {wait_seconds}s...')
            time.sleep(wait_seconds)
            continue

        print(f'  Error HTTP {response.status_code} en query "{query}"')
        return None

    print(f'  Reintentos agotados para query "{query}"')
    return None


def collect_unique_papers(queries: list[str]) -> list[dict]:
    """Recolecta papers únicos (deduplicados por paperId) ejecutando múltiples queries."""
    papers: list[dict] = []
    seen_paper_ids: set[str] = set()

    for query in queries:
        print(f'Consultando: "{query}"')
        result = fetch_papers_for_query(query)

        if result is None:
            print('  Falló, se omite esta query.')
            continue

        new_papers_count = 0
        for paper in result:
            paper_id = paper.get('paperId')
            if paper_id and paper_id not in seen_paper_ids:
                seen_paper_ids.add(paper_id)
                papers.append(paper)
                new_papers_count += 1

        print(f'  {len(result)} obtenidos, {new_papers_count} nuevos.')
        time.sleep(SLEEP_BETWEEN_QUERIES_SECONDS)

    return papers


def count_unique_authors(papers: list[dict]) -> int:
    """Cuenta autores únicos (por authorId) en la lista de papers."""
    author_ids = {
        author['authorId']
        for paper in papers
        for author in (paper.get('authors') or [])
        if author.get('authorId')
    }
    return len(author_ids)


# --- Ejecución ---
if RAW_PAPERS_PATH.exists():
    print(f'Ya existe {RAW_PAPERS_PATH}, cargando desde disco.')
    with open(RAW_PAPERS_PATH, encoding='utf-8') as f:
        papers = json.load(f)
else:
    papers = collect_unique_papers(PLN_QUERIES)

    with open(RAW_PAPERS_PATH, 'w', encoding='utf-8') as f:
        json.dump(papers, f, ensure_ascii=False, indent=2)
    print(f'\nGuardado en {RAW_PAPERS_PATH}')

print(f'\nTotal papers: {len(papers)}')
print(f'Autores únicos: {count_unique_authors(papers)}')

## 2. Construcción del grafo (15 pts)

**Modelado:**
- Nodos = autores (atributos: nombre, total de papers, años activos)
- Aristas = co-autorías
- Peso = número de papers compartidos entre el par

**Requisito:** mínimo 100 autores.

In [ ]:
# --- Cargar papers desde el JSON ---
with open(RAW_PAPERS_PATH, encoding='utf-8') as f:
    papers = json.load(f)

print(f'Papers cargados: {len(papers)}')


# --- Construcción del grafo ---
def build_collaboration_graph(papers: list[dict]) -> nx.Graph:
    """Construye un grafo no dirigido y ponderado de colaboraciones científicas.

    Nodos: autores (key = authorId), con atributos name, paper_count, years.
    Aristas: co-autorías. Peso = número de papers compartidos por el par.
    """
    graph = nx.Graph()

    for paper in papers:
        year = paper.get('year')
        authors_with_id = [
            author for author in (paper.get('authors') or [])
            if author.get('authorId')
        ]

        # Agregar o actualizar nodos
        for author in authors_with_id:
            author_id = author['authorId']
            if graph.has_node(author_id):
                graph.nodes[author_id]['paper_count'] += 1
                if year:
                    graph.nodes[author_id]['years'].add(year)
            else:
                graph.add_node(
                    author_id,
                    name=author.get('name', 'Unknown'),
                    paper_count=1,
                    years={year} if year else set(),
                )

        # Agregar o incrementar aristas entre cada par de co-autores
        unique_author_ids = list({author['authorId'] for author in authors_with_id})
        for author_id_a, author_id_b in combinations(unique_author_ids, 2):
            if graph.has_edge(author_id_a, author_id_b):
                graph[author_id_a][author_id_b]['weight'] += 1
            else:
                graph.add_edge(author_id_a, author_id_b, weight=1)

    return graph


G = build_collaboration_graph(papers)


# --- Estadísticas básicas ---
print('\n=== Estadísticas del grafo ===')
print(f'Nodos (autores):                  {G.number_of_nodes()}')
print(f'Aristas (colaboraciones únicas):  {G.number_of_edges()}')
print(f'Densidad:                         {nx.density(G):.6f}')

degrees = dict(G.degree())
average_degree = sum(degrees.values()) / G.number_of_nodes()
print(f'Grado promedio:                   {average_degree:.2f}')
print(f'Grado máximo:                     {max(degrees.values())}')

# Peso promedio de las aristas (cuántos papers comparte un par típico)
edge_weights = [data['weight'] for _, _, data in G.edges(data=True)]
average_weight = sum(edge_weights) / len(edge_weights) if edge_weights else 0
print(f'Peso promedio de aristas:         {average_weight:.2f}')

# Top 5 autores con más papers en el dataset
top_by_papers = sorted(
    G.nodes(data=True),
    key=lambda node: node[1]['paper_count'],
    reverse=True,
)[:5]

print('\nTop 5 autores por número de papers:')
for author_id, attrs in top_by_papers:
    print(f"  {attrs['name']:40s}  ({attrs['paper_count']} papers)")

## 3. Análisis (20 pts)

### 3a. Caminos mínimos

Encontrar rutas de colaboración entre autores. Usamos Dijkstra con peso invertido (`1 / peso`) para que más co-autorías signifique menor distancia.

In [ ]:
# --- Caminos mínimos ---
# El peso original es "nº de papers compartidos" (más es mejor).
# Para que Dijkstra interprete "más colaboraciones" como "más cerca",
# agregamos un atributo 'distance' = 1 / weight.

for _, _, edge_data in G.edges(data=True):
    edge_data['distance'] = 1.0 / edge_data['weight']

# Los algoritmos de caminos solo funcionan dentro de una componente conexa.
# Trabajamos sobre la componente gigante.
giant_component_nodes = max(nx.connected_components(G), key=len)
giant = G.subgraph(giant_component_nodes).copy()

print(f'Componente gigante: {giant.number_of_nodes()} nodos, '
      f'{giant.number_of_edges()} aristas')
print(f'(es decir, {giant.number_of_nodes() / G.number_of_nodes():.1%} '
      f'de todos los autores están conectados entre sí)')


# --- Ejemplo: camino más corto entre los dos top autores ---
top_authors = sorted(
    G.nodes(data=True),
    key=lambda node: node[1]['paper_count'],
    reverse=True,
)
author_source_id, author_source_attrs = top_authors[0]
author_target_id, author_target_attrs = top_authors[1]

if nx.has_path(G, author_source_id, author_target_id):
    path = nx.shortest_path(
        G, author_source_id, author_target_id, weight='distance',
    )
    path_names = [G.nodes[node]['name'] for node in path]

    print(f'\nCamino más corto de {author_source_attrs["name"]} '
          f'a {author_target_attrs["name"]}:')
    print('  ' + '  →  '.join(path_names))
    print(f'  ({len(path) - 1} salto(s))')
else:
    print(f'\n{author_source_attrs["name"]} y {author_target_attrs["name"]} '
          f'no están conectados (están en componentes distintas).')


# --- Estadísticas globales en la componente gigante ---
# Usamos peso uniforme para la métrica de "grados de separación".
print('\n=== Métricas globales (componente gigante, sin pesos) ===')
average_path_length = nx.average_shortest_path_length(giant)
diameter = nx.diameter(giant)
print(f'Longitud promedio de caminos: {average_path_length:.2f}')
print(f'Diámetro (camino más largo entre dos autores): {diameter}')

### 3b. Centralidad

Calculamos e interpretamos:
- **Degree centrality:** ¿quién tiene más colaboradores directos?
- **Betweenness centrality:** ¿quién actúa como puente entre subgrupos?
- **Closeness centrality:** ¿quién está más cerca de todos los demás en promedio?

In [ ]:
# --- Centralidad ---
# Calculamos las tres centralidades sobre la componente gigante.
# Trabajar sobre el grafo completo daría resultados engañosos: los nodos
# aislados tendrían closeness mal definida y betweenness = 0 artificial.

print(f'Calculando centralidad en la componente gigante '
      f'({giant.number_of_nodes()} nodos)...')

degree_centrality = nx.degree_centrality(giant)
betweenness_centrality = nx.betweenness_centrality(giant, weight='distance')
closeness_centrality = nx.closeness_centrality(giant, distance='distance')


def top_n_by_metric(
    metric_values: dict,
    graph: nx.Graph,
    n: int = 10,
) -> list[tuple[str, float]]:
    """Devuelve los top-N autores ordenados por la métrica dada."""
    ranked = sorted(metric_values.items(), key=lambda item: item[1], reverse=True)
    return [(graph.nodes[node_id]['name'], value) for node_id, value in ranked[:n]]


def print_centrality_ranking(title: str, interpretation: str, ranking: list) -> None:
    """Imprime un ranking de centralidad con título e interpretación."""
    print(f'\n=== {title} ===')
    print(f'({interpretation})')
    for name, value in ranking:
        print(f'  {name:40s}  {value:.4f}')


print_centrality_ranking(
    'Top 10 por Degree Centrality',
    'Cuántos colaboradores directos tiene cada autor (normalizado)',
    top_n_by_metric(degree_centrality, giant),
)

print_centrality_ranking(
    'Top 10 por Betweenness Centrality',
    'Qué tan frecuentemente un autor aparece en los caminos más cortos entre otros pares — actúa como "puente" entre subgrupos',
    top_n_by_metric(betweenness_centrality, giant),
)

print_centrality_ranking(
    'Top 10 por Closeness Centrality',
    'Qué tan cerca está un autor del resto de la red en promedio — los "más céntricos" llegan a todos rápido',
    top_n_by_metric(closeness_centrality, giant),
)

### 3c. Componentes conexas

Analizamos:
- Cuántas componentes hay
- Tamaño de la componente gigante
- Autores aislados (componentes de tamaño 1)
- Qué tan conectada está la red en general

In [ ]:
# --- Componentes conexas ---
components = list(nx.connected_components(G))
component_sizes = sorted((len(component) for component in components), reverse=True)

# Categorías por tamaño
isolated_authors = sum(1 for size in component_sizes if size == 1)
small_components = sum(1 for size in component_sizes if 2 <= size <= 5)
medium_components = sum(1 for size in component_sizes if 6 <= size <= 30)
large_components = sum(1 for size in component_sizes if size > 30)

print('=== Componentes conexas ===')
print(f'Total de componentes:               {len(components)}')
print(f'Componente gigante:                 {component_sizes[0]} nodos '
      f'({component_sizes[0] / G.number_of_nodes():.1%} del grafo)')
print(f'Autores aislados (componente=1):    {isolated_authors}')
print(f'Componentes pequeñas (2-5 nodos):   {small_components}')
print(f'Componentes medianas (6-30 nodos):  {medium_components}')
print(f'Componentes grandes (>30 nodos):    {large_components}')

print('\nTamaños de las 10 componentes más grandes:')
for index, size in enumerate(component_sizes[:10], start=1):
    print(f'  #{index}: {size} nodos')


# --- Inspección rápida de los autores aislados ---
isolated_node_ids = [n for n in G.nodes() if G.degree(n) == 0]
if isolated_node_ids:
    print(f'\nEjemplos de autores aislados ({len(isolated_node_ids)} en total):')
    for node_id in isolated_node_ids[:5]:
        print(f'  - {G.nodes[node_id]["name"]} '
              f'({G.nodes[node_id]["paper_count"]} paper(s))')

# --- Conectividad de la componente gigante ---
print('\n=== Conectividad de la componente gigante ===')
print(f'¿Es conexa?  {nx.is_connected(giant)}')
print(f'Coeficiente de clustering promedio: {nx.average_clustering(giant):.4f}')
print('(valores cercanos a 1 indican que los amigos de un autor también '
      'tienden a colaborar entre sí)')

### 3d. Detección de comunidades

Aplicamos el algoritmo de **Louvain** para identificar grupos de investigadores que colaboran intensamente entre sí. Interpretamos qué temas o líneas de investigación podrían representar.

In [ ]:
# --- Detección de comunidades (Louvain) ---
# Louvain optimiza la "modularidad": dentro de una comunidad las aristas
# son más densas que entre comunidades. Trabajamos sobre la componente
# gigante porque las comunidades pequeñas/aisladas no aportan información.

community_assignment = community_louvain.best_partition(
    giant,
    weight='weight',
    random_state=42,
)

# Cuántas comunidades y su tamaño
community_sizes = Counter(community_assignment.values())
sorted_communities = community_sizes.most_common()

print(f'=== Comunidades detectadas (Louvain) ===')
print(f'Total de comunidades: {len(community_sizes)}')
print(f'Modularidad del particionamiento: '
      f'{community_louvain.modularity(community_assignment, giant, weight="weight"):.4f}')
print('(modularidad > 0.3 indica estructura comunitaria clara)')

print('\nDistribución de tamaños:')
for community_id, size in sorted_communities[:10]:
    print(f'  Comunidad {community_id}: {size} autores')


# --- Para cada comunidad grande, mostrar sus autores más prominentes ---
# "Prominente" = autor con mayor degree dentro de la comunidad
def top_authors_in_community(
    graph: nx.Graph,
    members: list,
    top_n: int = 5,
) -> list[tuple[str, int]]:
    """Devuelve los top-N autores de una comunidad, ordenados por degree."""
    member_degrees = [(graph.nodes[node]['name'], graph.degree(node)) for node in members]
    return sorted(member_degrees, key=lambda x: x[1], reverse=True)[:top_n]


# Agrupar miembros por comunidad
members_by_community: dict[int, list] = defaultdict(list)
for node_id, community_id in community_assignment.items():
    members_by_community[community_id].append(node_id)

# Mostrar las 5 comunidades más grandes con sus autores destacados
print('\n=== Autores destacados de las 5 comunidades más grandes ===')
for community_id, _ in sorted_communities[:5]:
    members = members_by_community[community_id]
    top_authors = top_authors_in_community(giant, members)
    print(f'\nComunidad {community_id} ({len(members)} autores):')
    for name, degree in top_authors:
        print(f'  - {name} (degree={degree})')

# Guardar la asignación de comunidades en el grafo para usarla en la visualización
for node_id, community_id in community_assignment.items():
    giant.nodes[node_id]['community'] = community_id

## 4. Visualización (10 pts)

Generamos:
1. **Visualización completa del grafo** — todos los nodos y aristas, color por comunidad.
2. **Visualización destacando nodos importantes** — top autores por betweenness centrality resaltados.
3. _(Bonus)_ histograma de distribución de grados.

In [ ]:
# --- Preparación ---
FIGS_DIR = Path('figs')
FIGS_DIR.mkdir(exist_ok=True)

# Layout compartido por las dos visualizaciones (mismas posiciones)
print('Calculando layout spring (puede tardar 15-30 seg)...')
node_positions = nx.spring_layout(giant, seed=42, k=0.18, iterations=60)

# Atributos derivados que vamos a reutilizar
node_degrees = dict(giant.degree())
community_id_by_node = {n: giant.nodes[n]['community'] for n in giant.nodes()}


# --- Visualización 1: Grafo completo coloreado por comunidad ---
fig, ax = plt.subplots(figsize=(16, 12))

nx.draw_networkx_edges(
    giant, node_positions, ax=ax,
    alpha=0.15, width=0.5, edge_color='gray',
)
nx.draw_networkx_nodes(
    giant, node_positions, ax=ax,
    node_size=[20 + 3 * node_degrees[n] for n in giant.nodes()],
    node_color=[community_id_by_node[n] for n in giant.nodes()],
    cmap='tab10',
    alpha=0.85,
    linewidths=0.5,
    edgecolors='white',
)

ax.set_title(
    f'Red de colaboración en PLN — componente gigante\n'
    f'{giant.number_of_nodes()} autores · {giant.number_of_edges()} colaboraciones · '
    f'{len(set(community_id_by_node.values()))} comunidades (Louvain)',
    fontsize=14,
)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGS_DIR / 'grafo_completo.png', dpi=150, bbox_inches='tight')
plt.show()


# --- Visualización 2: Destacando los top-10 por betweenness ---
top_betweenness = sorted(
    betweenness_centrality.items(), key=lambda item: item[1], reverse=True,
)[:10]
top_ids = {node_id for node_id, _ in top_betweenness}
non_top_ids = [n for n in giant.nodes() if n not in top_ids]

fig, ax = plt.subplots(figsize=(16, 12))

nx.draw_networkx_edges(
    giant, node_positions, ax=ax,
    alpha=0.1, width=0.4, edge_color='gray',
)
nx.draw_networkx_nodes(
    giant, node_positions, ax=ax,
    nodelist=non_top_ids,
    node_size=15,
    node_color='lightgray',
    alpha=0.5,
)
nx.draw_networkx_nodes(
    giant, node_positions, ax=ax,
    nodelist=list(top_ids),
    node_size=500,
    node_color='crimson',
    alpha=0.95,
    edgecolors='black',
    linewidths=1.5,
)
nx.draw_networkx_labels(
    giant, node_positions, ax=ax,
    labels={n: giant.nodes[n]['name'] for n in top_ids},
    font_size=10,
    font_weight='bold',
)

ax.set_title(
    'Top 10 autores por Betweenness Centrality\n'
    '(los "puentes" más importantes entre subgrupos)',
    fontsize=14,
)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGS_DIR / 'grafo_top_betweenness.png', dpi=150, bbox_inches='tight')
plt.show()


# --- Visualización 3 (bonus): Distribución de grados en escala log-log ---
all_degrees = [d for _, d in G.degree()]
degree_frequency = Counter(all_degrees)
degree_values = sorted(d for d in degree_frequency if d > 0)  # log requiere >0
frequencies = [degree_frequency[d] for d in degree_values]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(degree_values, frequencies, alpha=0.7, s=40, color='steelblue')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Grado del nodo (escala log)')
ax.set_ylabel('Número de nodos con ese grado (escala log)')
ax.set_title(
    'Distribución de grados de la red\n'
    '(una línea aproximadamente recta sugiere una red libre de escala)',
    fontsize=13,
)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(FIGS_DIR / 'distribucion_grados.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Visualización interactiva con pyvis ---
# Genera un HTML que se abre en el navegador con zoom, pan, hover y arrastre.
# Excelente para mostrar en la presentación.
from pyvis.network import Network
import matplotlib

interactive_net = Network(
    height='800px',
    width='100%',
    bgcolor='#ffffff',
    font_color='#222222',
    notebook=False,
)
interactive_net.force_atlas_2based(
    gravity=-50, central_gravity=0.01, spring_length=120, spring_strength=0.08,
)

# Paleta de colores por comunidad
palette = matplotlib.colormaps.get_cmap('tab10')
unique_communities = sorted(set(community_id_by_node.values()))
color_by_community = {
    cid: matplotlib.colors.rgb2hex(palette(i % 10))
    for i, cid in enumerate(unique_communities)
}

# Agregar nodos
for node_id in giant.nodes():
    attrs = giant.nodes[node_id]
    degree = giant.degree(node_id)
    interactive_net.add_node(
        str(node_id),
        label=attrs['name'],
        title=(
            f"{attrs['name']}\n"
            f"Papers: {attrs['paper_count']}\n"
            f"Colaboradores: {degree}\n"
            f"Comunidad: {attrs['community']}"
        ),
        color=color_by_community[attrs['community']],
        size=10 + degree * 0.5,
    )

# Agregar aristas
for source, target, edge_data in giant.edges(data=True):
    interactive_net.add_edge(
        str(source), str(target),
        value=edge_data['weight'],
        title=f"{edge_data['weight']} paper(s) en común",
    )

interactive_html_path = FIGS_DIR / 'grafo_interactivo.html'
interactive_net.write_html(str(interactive_html_path), notebook=False, open_browser=False)
print(f'Visualización interactiva guardada en: {interactive_html_path}')
print('Ábrelo con doble click en el archivo (se abrirá en tu navegador).')

## 5. Conclusiones

### Sobre la red de PLN

A partir de 200 papers reales de Semantic Scholar logramos reconstruir una red de colaboración con **933 autores únicos** y **8,130 colaboraciones** distintas, superando con holgura el mínimo de 100 autores requerido.

### Hallazgos principales

1. **La red tiene la propiedad de mundo pequeño.** La longitud promedio de los caminos en la componente gigante es de **2.94 saltos** y su diámetro es de **7**. Esto significa que cualquier par de investigadores del núcleo de PLN está separado por menos de 3 colaboraciones intermedias en promedio — un patrón clásico de redes sociales reales.

2. **Existe una componente gigante claramente dominante.** El **40.7%** de todos los autores (380) forman un solo subgrafo conectado, mientras que el resto se distribuye en 118 componentes más pequeñas: 80 micro-componentes de 2-5 autores, 28 medianas, 2 grandes (35 y 28 nodos) y 9 autores aislados. Esto refleja la estructura típica de un campo científico maduro: un núcleo activo de colaboradores frecuentes, rodeado de pequeños equipos satélite.

3. **La red es altamente "triangulada".** El coeficiente de clustering promedio de la componente gigante es **0.901**, indicando que los colaboradores de un autor también tienden a colaborar entre sí. Es una propiedad esperable en coautoría académica: los papers con múltiples autores generan triángulos completos.

4. **Estructura comunitaria fuerte.** El algoritmo de Louvain detectó **7 comunidades** con una **modularidad de 0.6319** (muy por encima del umbral 0.3 que indica estructura clara). Las dos comunidades más grandes corresponden a:
   - **Comunidad NLLB (98 autores):** liderada por M. Costa-jussà, Wenzek, Barrault, Gao, Schwenk. Refleja el proyecto _No Language Left Behind_ de Meta AI, un esfuerzo masivo de traducción multilingüe.
   - **Comunidad MT clásica (77 autores):** liderada por Philipp Koehn. Concentra a los autores asociados a las competencias WMT y al sistema Moses, los referentes históricos de la traducción automática estadística.

5. **Autores puente identificados.** La centralidad de intermediación (betweenness) reveló que **Markus Freitag** y **C. Federmann** actúan como puentes entre subcomunidades, conectando el cluster NLLB con el de WMT pese a no tener el mayor número de colaboradores individuales.

### Limitaciones

- El dataset proviene únicamente de búsquedas por palabra clave en Semantic Scholar; un autor cuyos papers no contengan ninguno de los 8 términos consultados quedaría fuera, aunque sea relevante en PLN.
- Solo dos de las ocho queries planeadas lograron pasar el rate limit de la API sin key. Con una API key oficial podríamos triplicar el tamaño del dataset.
- El peso de las aristas (papers compartidos) no refleja el _orden_ de autoría ni el impacto del paper.

### Trabajo futuro

- Solicitar la API key de Semantic Scholar para ampliar la recolección.
- Enriquecer las aristas con métricas adicionales: número de citas conjuntas, distancia temporal entre colaboraciones.
- Comparar la red de PLN con la de otra subárea de computación para identificar patrones estructurales distintivos.